# Technical Metrics Construct Validation

## Purpose
This notebook implements construct validation for the three technical DeepEval 
metrics in the IHAEF framework: Answer Relevancy (AR), Contextual Relevancy (CR), 
and Faithfulness (F). Validation tests whether each metric discriminates between 
construct-positive and construct-negative cases using a positive/negative control 
paradigm with similarity-filtered negatives.

## Design
- **Dataset:** `goldens_final.csv` (102 rows; synthesizer-generated input/expected_output 
  paired with HIA's actual retrieved context per row)
- **Embedding model:** Azure OpenAI `text-embedding-3-small` (same as HIA's RAG pipeline)
- **Positive controls:** rows with their original (input, expected_output, context) triples, 
  trimmed per metric by removing the bottom 10% of construct-relevant cosine similarities 
  to address HIA retrieval contamination
- **Negative controls:** shuffled pairings filtered to fall within the 5th–25th percentile 
  band of trimmed positive cosines, ensuring non-trivial but distinguishable negatives
- **Evaluation:** DeepEval metrics scored on positive and negative cases; ROC AUC computed 
  with positive=1, negative=0

## Hypotheses (pre-committed)
For each metric:
- **H0:** AUC ≤ 0.8 (insufficient discrimination)
- **H1:** AUC > 0.8 (sufficient discrimination per Hosmer & Lemeshow, 2013)

## Notes
- `actual_output` is set to `expected_output` to isolate metric behavior from system behavior 
  (construct validation tests the metric, not the HIA pipeline)
- Negative-control sampling involves randomized candidate selection; exact counts may vary 
  across runs but band-based filtering ensures consistency in negative quality
- F validation was conducted but its results require methodological interpretation 
  (see diagnostic section)

In [41]:
import os
import re
import ast
import time
import random
import pickle
import numpy as np
import pandas as pd
from rich import print
from deepeval import evaluate
from dotenv import load_dotenv
from openai import AzureOpenAI
from deepeval.models import AzureOpenAIModel
from deepeval.test_case import LLMTestCase
from sklearn.metrics.pairwise import cosine_similarity
from deepeval.metrics import AnswerRelevancyMetric, ContextualRelevancyMetric, FaithfulnessMetric

In [3]:
data = pd.read_csv("../data/goldens_final.csv", encoding='utf-8-sig')
print(data.head())

user_input  \
0  Where can I find info about support orgs and s...   
1  How do LOS and Red Cross websites differ in in...   
2  If UM lost access to all support orgs, what ne...   
3  How do limited rights, lack of benefits, and h...   
4  Analyze how limited rights to social benefits ...   

                                     expected_output  \
0  You can find information about support organiz...   
1  The LOS website provides detailed information ...   
2  If undocumented migrants (UM) lose access to a...   
3  Limited rights, lack of benefits, and housing ...   
4  Limited rights to social benefits and health i...   

                                          bot_output  \
0  For information about support organizations an...   
1  Based on the provided documents, there is no s...   
2  If an Unaccompanied Minor (UM) loses access to...   
3  I don't have the right information to answer y...   
4  Limited rights to social benefits and health i...   

                                             context  
0  ["Document: What are the steps I should take t...  
1  ["Document: Remaining Undocumented If you rema...  
2  ["Document: I need medical treatment in the Ne...  
3  ["Document: My host family/friends/family wher...  
4  ["Document: Do I need health insurance?\n\nFor...

## Embedding + cosine utility

In [4]:
# Client setup
load_dotenv()
key = os.getenv("azure_subscription_key")
api_version = "2024-12-01-preview"

endpoint = "https://510-ai-research.cognitiveservices.azure.com/"
model_name = "text-embedding-3-small"
deployment = "text-embedding-3-small"

client = AzureOpenAI(
    api_version=api_version,
    azure_endpoint=endpoint,
    api_key=key,
)

In [5]:
# Batch embedding
def embed_texts(texts, batch_size=50):
  """Embed a list of strings."""
  all_vectors = []
  for i in range(0, len(texts), batch_size):
      batch = texts[i : i + batch_size]
      response = client.embeddings.create(input=batch, model=deployment)
      all_vectors.extend([item.embedding for item in response.data])
  return np.array(all_vectors)

In [6]:
# aggregate a context (list of chunks) to one vector
def embed_context(chunks):
  """Mean-pool embeddings of all chunks in one context."""
  vectors = embed_texts(chunks)
  return vectors.mean(axis=0)

In [7]:
# cosine between two vectors
def cosine(vec_a, vec_b):
    """Cosine similarity between two 1D vectors. Returns a float."""
    return cosine_similarity(vec_a.reshape(1, -1), vec_b.reshape(1, -1))[0, 0]

In [8]:
# Sanity check
for i in [0, 9, 50]:
    inp_vec = embed_texts([data['user_input'][i]])[0]
    exp_vec = embed_texts([data['expected_output'][i]])[0]
    ctx_vec = embed_context(ast.literal_eval(data['context'][i]))

    print(f"Row {i}:")
    print(f"  cosine(input, expected_output) = {cosine(inp_vec, exp_vec):.3f}")
    print(f"  cosine(input, context)         = {cosine(inp_vec, ctx_vec):.3f}")
    print(f"  cosine(expected_output, context) = {cosine(exp_vec, ctx_vec):.3f}")
    print()

Row 0:

cosine(input, expected_output) = 0.655

cosine(input, context)         = 0.547

cosine(expected_output, context) = 0.721

Row 9:

...

cosine(expected_output, context) = 0.610

Row 50:

cosine(input, expected_output) = 0.583

cosine(input, context)         = 0.551

cosine(expected_output, context) = 0.726

### Calculate summary stats for all cosines
This is to ensure strong positive pairings.

In [9]:
# For AR
cos = []
for i in range(len(data)):
    inp_vec = embed_texts([data['user_input'][i]])[0]
    exp_vec = embed_texts([data['expected_output'][i]])[0]
    cos.append(cosine(inp_vec, exp_vec))

In [10]:
print(np.mean(cos))
print(np.min(cos))
print(np.max(cos))
print(np.median(cos))
print(np.percentile(cos, 25))
print(np.percentile(cos, 5))

0.7680429861861517

0.5807913856194041

0.9073471428550977

0.7707437884656998

0.7278871294396059

0.6546518335228771

Min > 0.58 indicates no broken pairings. Every row has a genuinely well-matched (input, expected_output) pair. 

Mean approx. the same as median, no extreme tail. The synthesizer produces consistent quality.

We will use the 5-25th percentile band to filter AR negatives - a shuffled `expected_output` from another row counts as a negative-control candidate only if its cosine to the input falls in [0.655, 0.728].

Example:
- A candidate negative with cosine > 0.728 (above the 25th percentile of positives) is "too similar" — it might actually be a valid answer to the question. Reject as a negative candidate.
- A candidate negative with cosine < 0.655 (below the 5th percentile of positives) is "trivially dissimilar" — it's the easy-negative case that artificially inflates AUC. Reject.
- A candidate negative with cosine in [0.655, 0.728] is in the same similarity range as the weakest genuine positives. Hard enough to be a meaningful test, far enough from positive-territory that the label is defensible. Accept.

In [11]:
# For CR
cos = []
for i in range(len(data)):
    inp_vec = embed_texts([data['user_input'][i]])[0]
    exp_vec = embed_context(ast.literal_eval(data['context'][i]))
    cos.append(cosine(inp_vec, exp_vec))

print(np.mean(cos))
print(np.min(cos))
print(np.max(cos))
print(np.median(cos))
print(np.percentile(cos, 25))
print(np.percentile(cos, 5))

0.5516796887049633

0.10001649455510356

0.763116238834693

0.5620895153462901

0.4980506948562605

0.3831293513857333

In [12]:
# For F
cos = []
for i in range(len(data)):
    inp_vec = embed_texts([data['expected_output'][i]])[0]
    exp_vec = embed_context(ast.literal_eval(data['context'][i]))
    cos.append(cosine(inp_vec, exp_vec))

print(np.mean(cos))
print(np.min(cos))
print(np.max(cos))
print(np.median(cos))
print(np.percentile(cos, 25))
print(np.percentile(cos, 5))

0.6377176185272847

0.06920770943780243

0.8022516652984514

0.6544160551779539

0.6054223887123307

0.43875837258331934

CR and F have ugly minimums. A min of 0.10 for CR means at least one row has a question and a retrieved context that are essentially semantically unrelated. F min of 0.07 means at least one row has an expected_output that doesn't resemble its retrieved context. This means that for some questions, HIA pulls irrelevant or only loosely related chunks.

Looking at the distributions, I decided it is better to recompute the band on a trimmed set (drop bottom 10%) and calculate the new 10th - 25th band. 

## Determine trimmed positive set

In [13]:
input_vecs = embed_texts(list(data['user_input']))
exp_vecs = embed_texts(list(data['expected_output']))
ctx_vecs = np.array([
    embed_context(ast.literal_eval(data['context'][i])) 
    for i in range(len(data))
])

In [14]:
# Per-metric positive cosines, trimming, bands
def positive_cosines(anchor_vecs, target_vecs):
    return np.array([cosine(anchor_vecs[i], target_vecs[i]) for i in range(len(anchor_vecs))])

ar_cos = positive_cosines(input_vecs, exp_vecs)
cr_cos = positive_cosines(input_vecs, ctx_vecs)
f_cos = positive_cosines(exp_vecs, ctx_vecs)

We apply per-metric percentile cutoff: for each metric, the bottom 10% of its positives (i.e., a pair that is genuinenly relevant). The cutoff value will be different for each metric because each distribution has a different shape. This ensures symmetric data treatment accross metrics with differing baseline distributions.

In [15]:
def trim_and_band(cosines, drop_pct=10):
    """Drop bottom drop_pct% of rows; return kept_indices and band (5th, 25th pct of kept)."""
    cutoff = np.percentile(cosines, drop_pct)
    kept = np.where(cosines > cutoff)[0]
    kept_cosines = cosines[kept]
    band = (np.percentile(kept_cosines, 5), np.percentile(kept_cosines, 25))
    return kept, band

### Compute bands (5th–25th pct of trimmed)

In [16]:
ar_kept, ar_band = trim_and_band(ar_cos)
cr_kept, cr_band = trim_and_band(cr_cos)
f_kept, f_band = trim_and_band(f_cos)

print(f"AR: {len(ar_kept)} positives, band {ar_band}")
print(f"CR: {len(cr_kept)} positives, band {cr_band}")
print(f"F:  {len(f_kept)} positives, band {f_band}")

AR: 91 positives, band (np.float64(0.7056462111636336), np.float64(0.7392090884620319))

CR: 91 positives, band (np.float64(0.4412993211893975), np.float64(0.5263359427030834))

F:  91 positives, band (np.float64(0.5608236983766337), np.float64(0.6280192583370308))

### Build Negatives

In [17]:
# Build negatives
def find_negatives(kept_indices, anchor_vecs, target_vecs, band, max_retries=200):
    """For each kept positive, find a negative partner with cosine in band."""
    pairs = []
    failed = []
    kept_set = set(kept_indices.tolist())
    for i in kept_indices:
        for attempt in range(max_retries):
            j = random.choice(list(kept_set - {i}))
            c = cosine(anchor_vecs[i], target_vecs[j])
            if band[0] <= c <= band[1]:
                pairs.append((int(i), int(j), float(c)))
                break
        else:
            failed.append(int(i))
    return pairs, failed

In [18]:
ar_negs, ar_failed = find_negatives(ar_kept, input_vecs, exp_vecs, ar_band)
cr_negs, cr_failed = find_negatives(cr_kept, input_vecs, ctx_vecs, cr_band)
f_negs, f_failed = find_negatives(f_kept, exp_vecs, ctx_vecs, f_band)

print(f"AR negatives: {len(ar_negs)} found, {len(ar_failed)} failed")
print(f"CR negatives: {len(cr_negs)} found, {len(cr_failed)} failed")
print(f"F  negatives: {len(f_negs)} found, {len(f_failed)} failed")

AR negatives: 19 found, 72 failed

CR negatives: 83 found, 8 failed

F  negatives: 83 found, 8 failed

CR and F are mostly fine — losing 10% is acceptable, as there will be have 80+ negatives per metric.

AR is broken. The AR band is so narrow (0.033 wide, way up at 0.7+) that random shuffled pairs almost never land in it. This means that AR positive distribution is so tight that the 5th–25th percentile band is in a region that random text simply doesn't reach.

Since we introduced the bottom 10% cutoff due to CR and F having positives with very low cosines, while AR's positives were clean and tight, we will skip this remedy for AR and keep all 102 AR positives. The band will be recomputed as 5th-25th percentile of all the data.

In [19]:
ar_band = (0.655, 0.728)
all_indices = np.arange(len(data))
ar_negs, ar_failed = find_negatives(all_indices, input_vecs, exp_vecs, ar_band)
print(f"AR negatives: {len(ar_negs)} found, {len(ar_failed)} failed")

AR negatives: 39 found, 63 failed

For AR, 44 positives + 44 negatives = 88 cases.

## Run DeepEval

In [23]:
# Build test cases per metric, with labels stored separately
def build_ar_cases(positive_idx_list, neg_pairs, data):
    """Returns (cases, labels) for Answer Relevancy."""
    cases = []
    labels = []
    # Positives
    for i in positive_idx_list:
        cases.append(LLMTestCase(
            input=data['user_input'][i],
            actual_output=data['expected_output'][i],
        ))
        labels.append(1)
    # Negatives
    for i, j, _ in neg_pairs:
        cases.append(LLMTestCase(
            input=data['user_input'][i],
            actual_output=data['expected_output'][j],
        ))
        labels.append(0)
    return cases, labels


def build_cr_cases(positive_idx_list, neg_pairs, data):
    """Returns (cases, labels) for Contextual Relevancy."""
    cases = []
    labels = []
    for i in positive_idx_list:
        cases.append(LLMTestCase(
            input=data['user_input'][i],
            actual_output=data['expected_output'][i],  # required field, not scored by CR
            retrieval_context=ast.literal_eval(data['context'][i]),
        ))
        labels.append(1)
    for i, j, _ in neg_pairs:
        cases.append(LLMTestCase(
            input=data['user_input'][i],
            actual_output=data['expected_output'][i],
            retrieval_context=ast.literal_eval(data['context'][j]),
        ))
        labels.append(0)
    return cases, labels


def build_f_cases(positive_idx_list, neg_pairs, data):
    """Returns (cases, labels) for Faithfulness."""
    cases = []
    labels = []
    for i in positive_idx_list:
        cases.append(LLMTestCase(
            input=data['user_input'][i],  # required field
            actual_output=data['expected_output'][i],
            retrieval_context=ast.literal_eval(data['context'][i]),
        ))
        labels.append(1)
    for i, j, _ in neg_pairs:
        cases.append(LLMTestCase(
            input=data['user_input'][i],
            actual_output=data['expected_output'][i],
            retrieval_context=ast.literal_eval(data['context'][j]),
        ))
        labels.append(0)
    return cases, labels


# Extract positive indices from neg_pairs (only positives that found a negative partner)
ar_positive_indices = [i for i, j, c in ar_negs]
cr_positive_indices = [i for i, j, c in cr_negs]
f_positive_indices  = [i for i, j, c in f_negs]

ar_cases, ar_labels = build_ar_cases(ar_positive_indices, ar_negs, data)
cr_cases, cr_labels = build_cr_cases(cr_positive_indices, cr_negs, data)
f_cases, f_labels   = build_f_cases(f_positive_indices, f_negs, data)

print(f"AR: {len(ar_cases)} cases ({sum(ar_labels)} positive, {len(ar_labels) - sum(ar_labels)} negative)")
print(f"CR: {len(cr_cases)} cases ({sum(cr_labels)} positive, {len(cr_labels) - sum(cr_labels)} negative)")
print(f"F:  {len(f_cases)} cases ({sum(f_labels)} positive, {len(f_labels) - sum(f_labels)} negative)")

AR: 78 cases (39 positive, 39 negative)

CR: 166 cases (83 positive, 83 negative)

F:  166 cases (83 positive, 83 negative)

In [ ]:
# CUSTOM MODEL
endpoint = "https://510-ai-research.openai.azure.com/"
model = "gpt-4.1"
deployment = "gpt-4.1-students"
# loading variables from .env file
subscription_key = os.getenv("azure_subscription_key")
api_version = "2024-12-01-preview"
# model
custom_model = AzureOpenAIModel(
  model=deployment,
  api_key=subscription_key,
  azure_endpoint=endpoint,
  api_version=api_version,
  deployment_name=deployment
)

ar_metric = AnswerRelevancyMetric(model=custom_model, async_mode=False)
print("Running AR evaluation...")
ar_results = evaluate(test_cases=ar_cases, metrics=[ar_metric])

with open('ar_results.pkl', 'wb') as f:
    pickle.dump(ar_results, f) # saving raw results to disk in case notebook crashes

Running AR evaluation...

✨ You're running DeepEval's latest Answer Relevancy Metric! (using gpt-4.1-students (Azure), strict=False, 
async_mode=True)...

c:\Users\dari\conda\envs\hia\Lib\site-packages\rich\live.py:256: UserWarning: install "ipywidgets" for Jupyter 
support
  warnings.warn('install "ipywidgets" for Jupyter support')



Metrics Summary

  - ✅ Answer Relevancy (score: 1.0, threshold: 0.5, strict: False, evaluation model: gpt-4.1-students (Azure), reason: Great job! The score is 1.00 because the answer was fully relevant and addressed the input without any irrelevant statements., error: None)

For test case:

  - input: If a migrant evades detection for 18 months after a Dublin claim, what happens next?
  - actual output: If a migrant with a Dublin claim manages to stay under the radar for 18 months, the Dublin claim expires. After this period, they can request asylum in the Netherlands.
  - expected output: None
  - context: None
  - retrieval context: None


Metrics Summary

  - ✅ Answer Relevancy (score: 1.0, threshold: 0.5, strict: False, evaluation model: gpt-4.1-students (Azure), reason: Great job! The score is 1.00 because the output was fully relevant and addressed the input without any irrelevant statements., error: None)

For test case:

  - input: Which NL hospitals have CAK contracts for n

⚠ WARNING: No hyperparameters logged.
» ]8;id=446334;https://deepeval.com/docs/evaluation-prompts\Log hyperparameters]8;;\ to attribute prompts and models to your test runs.

================================================================================

✓ Evaluation completed 🎉! (time taken: 24.42s | token cost: None)
» Test Results (78 total tests):
   » Pass Rate: 92.31% | Passed: 72 | Failed: 6

 ================================================================================ 

» Want to share evals with your team, or a place for your test cases to live? ❤️ 🏡
  » Run 'deepeval view' to analyze and save testing results on Confident AI.

In [ ]:
def extract_scores_and_labels(eval_result, labels):
    """
    Returns two aligned lists: scores and labels, ordered by original case index.
    """
    n_cases = len(labels)
    scores = [None] * n_cases
    for tr in eval_result.test_results:
        # tr.name is "test_case_N"
        match = re.match(r'test_case_(\d+)', tr.name)
        if not match:
            raise ValueError(f"Unexpected name format: {tr.name}")
        original_idx = int(match.group(1))
        scores[original_idx] = tr.metrics_data[0].score
    
    if any(s is None for s in scores):
        missing = [i for i, s in enumerate(scores) if s is None]
        raise ValueError(f"Missing scores for indices: {missing}")
    
    return scores, labels

ar_scores, ar_labels_aligned = extract_scores_and_labels(ar_results, ar_labels)
print(f"AR: {len(ar_scores)} scores extracted")
print(f"  Positives (label=1): {[s for s, l in zip(ar_scores, ar_labels_aligned) if l == 1][:5]}...")
print(f"  Negatives (label=0): {[s for s, l in zip(ar_scores, ar_labels_aligned) if l == 0][:5]}...")

AR: 78 scores extracted

Positives (label=1): [1.0, 1.0, 1.0, 1.0, 1.0]...

Negatives (label=0): [1.0, 0.8571428571428571, 1.0, 0.16666666666666666, 0.0]...

Two negative cases scored perfect 1.0. That means AR judged a shuffled (input from row i, expected_output from row j) pair as fully relevant. These are the cases AR can't distinguish. It might be because some shuffled negatives ended up being topically close enough that the answer happened to address the question reasonably well anyway (your band did pick "non-trivial" negatives, after all)

In [42]:
def evaluate_in_batches(cases, metric, batch_size=20, sleep_sec=30):
    all_results = []
    for i in range(0, len(cases), batch_size):
        batch = cases[i:i+batch_size]
        print(f"Batch {i//batch_size + 1}: cases {i} to {i+len(batch)-1}")
        result = evaluate(test_cases=batch, metrics=[metric])
        all_results.extend(result.test_results)
        if i + batch_size < len(cases):
            print(f"Sleeping {sleep_sec}s...")
            time.sleep(sleep_sec)
    return all_results

In [44]:
cr_metric = ContextualRelevancyMetric(model=custom_model, async_mode=False)
print("Running CR evaluation...")
cr_results = evaluate_in_batches(cr_cases, metric=cr_metric)
with open('cr_results.pkl', 'wb') as f:
    pickle.dump(cr_results, f)

Running CR evaluation...

Batch 1: cases 0 to 19

✨ You're running DeepEval's latest Contextual Relevancy Metric! (using gpt-4.1-students (Azure), strict=False, 
async_mode=True)...

c:\Users\dari\conda\envs\hia\Lib\site-packages\rich\live.py:256: UserWarning: install "ipywidgets" for Jupyter 
support
  warnings.warn('install "ipywidgets" for Jupyter support')



Metrics Summary

  - ❌ Contextual Relevancy (score: 0.09433962264150944, threshold: 0.5, strict: False, evaluation model: gpt-4.1-students (Azure), reason: The score is 0.09 because, as quoted, 'The statement lists required facilities at a crisis emergency shelter, but does not address how undocumented minors access emergency shelter during extreme weather in non-participating municipalities.' and the relevant statements only provide general information like 'Undocumented persons do not have a right to housing, work or government subsidies.' without directly answering the input question., error: None)

For test case:

  - input: How do undocumented minors access emergency shelter during extreme weather in non-participating municipalities?
  - actual output: Undocumented minors can access emergency shelter during extreme weather through the Cold Weather Regulation, which municipalities are legally required to organize when temperatures drop below 0°C, even if they do not have a perman

...

Batch 9: cases 160 to 165

✨ You're running DeepEval's latest Contextual Relevancy Metric! (using gpt-4.1-students (Azure), strict=False, 
async_mode=True)...



Metrics Summary

  - ❌ Contextual Relevancy (score: 0.01098901098901099, threshold: 0.5, strict: False, evaluation model: gpt-4.1-students (Azure), reason: The score is 0.01 because nearly all statements are irrelevant, such as 'The statement defines unaccompanied minors arriving in the Netherlands, which is unrelated to the Red Cross's distribution of clothing,' and only one statement mentions 'Clothes (seasonal) young and old' at crisis shelters, which does not directly answer the input question., error: None)

For test case:

  - input: Does the Red Cross distribute clothing directly to individuals, or only to shelter locations?
  - actual output: The Red Cross only provides clothing to shelter locations, not directly to individuals.
  - expected output: None
  - context: None
  - retrieval context: ['Document: What are the steps I should take to register a child travelling alone from Ukraine?  \n\nUnaccompanied minor foreingers are children who are under the age of 18 when they a

⚠ WARNING: No hyperparameters logged.
» ]8;id=204613;https://deepeval.com/docs/evaluation-prompts\Log hyperparameters]8;;\ to attribute prompts and models to your test runs.

================================================================================

✓ Evaluation completed 🎉! (time taken: 20.5s | token cost: None)
» Test Results (6 total tests):
   » Pass Rate: 0.0% | Passed: 0 | Failed: 6

 ================================================================================ 

» Want to share evals with your team, or a place for your test cases to live? ❤️ 🏡
  » Run 'deepeval view' to analyze and save testing results on Confident AI.

In [45]:
f_metric  = FaithfulnessMetric(model=custom_model, async_mode=False)
print("Running F evaluation...")
f_results = evaluate_in_batches(f_cases, metric=f_metric)
with open('f_results.pkl', 'wb') as f:
    pickle.dump(f_results, f)

Running F evaluation...

Batch 1: cases 0 to 19

✨ You're running DeepEval's latest Faithfulness Metric! (using gpt-4.1-students (Azure), strict=False, 
async_mode=True)...

c:\Users\dari\conda\envs\hia\Lib\site-packages\rich\live.py:256: UserWarning: install "ipywidgets" for Jupyter 
support
  warnings.warn('install "ipywidgets" for Jupyter support')



Metrics Summary

  - ✅ Faithfulness (score: 1.0, threshold: 0.5, strict: False, evaluation model: gpt-4.1-students (Azure), reason: The score is 1.00 because there are no contradictions listed, indicating the actual output aligns perfectly with the retrieval context. Great job maintaining faithfulness!, error: None)

For test case:

  - input: Where can I find info about support orgs and shelters for UM, incl. local options?
  - actual output: You can find information about support organizations and shelters for undocumented migrants (UM), including local options, on the Stichting LOS website: https://www.stichtinglos.nl/content/organisaties and for shelters: https://www.stichtinglos.nl/noodopvang. Additionally, the Red Cross Helpful Information website provides details about local organizations in different cities: https://helpfulinformation.redcross.nl/.
  - expected output: None
  - context: None
  - retrieval context: ['Document: What are the steps I should take to register a chi

...

Batch 9: cases 160 to 165

✨ You're running DeepEval's latest Faithfulness Metric! (using gpt-4.1-students (Azure), strict=False, 
async_mode=True)...



Metrics Summary

  - ✅ Faithfulness (score: 1.0, threshold: 0.5, strict: False, evaluation model: gpt-4.1-students (Azure), reason: The score is 1.00 because there are no contradictions listed, indicating the actual output aligns perfectly with the retrieval context. Great job maintaining faithfulness!, error: None)

For test case:

  - input: How can Sympany's partnership and HIA city lists help locate nearby clothing donation containers?
  - actual output: Sympany's partnership with the Red Cross allows you to find nearby clothing donation containers using their website: https://www.sympany.nl/. Additionally, the HIA city lists (https://helpfulinformation.redcross.nl/) provide location-specific options, helping you identify the closest donation points in your area.
  - expected output: None
  - context: None
  - retrieval context: ['Document: Red Cross The Red Cross focuses on humanitarian assistance, they offer for example; first aid, emergency food support and psychosocial suppor

⚠ WARNING: No hyperparameters logged.
» ]8;id=735866;https://deepeval.com/docs/evaluation-prompts\Log hyperparameters]8;;\ to attribute prompts and models to your test runs.

================================================================================

✓ Evaluation completed 🎉! (time taken: 26.09s | token cost: None)
» Test Results (6 total tests):
   » Pass Rate: 100.0% | Passed: 6 | Failed: 0

 ================================================================================ 

» Want to share evals with your team, or a place for your test cases to live? ❤️ 🏡
  » Run 'deepeval view' to analyze and save testing results on Confident AI.

### Extract scores and labels

In [51]:
def extract_from_batches(results_list, batch_size, labels):
    """Extract scores from batched evaluation results."""
    n_cases = len(labels)
    scores = [None] * n_cases
    n_batches = (n_cases + batch_size - 1) // batch_size
    
    cursor = 0
    for batch_idx in range(n_batches):
        batch_start = batch_idx * batch_size
        batch_end = min(batch_start + batch_size, n_cases)
        batch_n = batch_end - batch_start
        
        batch_entries = results_list[cursor : cursor + batch_n]
        cursor += batch_n
        
        for tr in batch_entries:
            match = re.match(r'test_case_(\d+)', tr.name)
            local_idx = int(match.group(1))
            original_idx = batch_start + local_idx
            scores[original_idx] = tr.metrics_data[0].score
    
    if any(s is None for s in scores):
        missing = [i for i, s in enumerate(scores) if s is None]
        raise ValueError(f"Missing scores for indices: {missing}")
    
    return scores, labels

cr_scores, cr_labels_aligned = extract_from_batches(cr_results, batch_size=20, labels=cr_labels)
print(f"CR: {len(cr_scores)} scores extracted, no missing")

CR: 166 scores extracted, no missing

In [53]:
f_scores, f_labels_aligned = extract_from_batches(f_results, batch_size=20, labels=f_labels)
print(f"F: {len(f_scores)} scores extracted, no missing")

F: 166 scores extracted, no missing

In [55]:
def quick_check(scores, labels, name):
    s = np.array(scores)
    l = np.array(labels)
    pos = s[l == 1]
    neg = s[l == 0]
    print(f"{name}: positives mean={pos.mean():.3f}, neg mean={neg.mean():.3f}")
    print(f"      positives min={pos.min():.3f}, max={pos.max():.3f}")
    print(f"      negatives min={neg.min():.3f}, max={neg.max():.3f}")

quick_check(ar_scores, ar_labels_aligned, "AR")
quick_check(cr_scores, cr_labels_aligned, "CR")
quick_check(f_scores, f_labels_aligned, "F ")

AR: positives mean=0.977, neg mean=0.798

positives min=0.556, max=1.000

negatives min=0.000, max=1.000

CR: positives mean=0.262, neg mean=0.161

positives min=0.000, max=0.891

negatives min=0.000, max=0.769

F : positives mean=0.941, neg mean=0.968

positives min=0.400, max=1.000

negatives min=0.000, max=1.000

- AR — looks healthy
Positive mean 0.977 vs negative mean 0.798. AR is discriminating, though negatives are scoring higher than ideal.
- CR — the metric is weak, but discriminating
Positive mean 0.262, negative mean 0.161. Positives are higher than negatives, but both means are very low. With 20+ chunks per context and a specific input, most chunks won't be relevant, even for positive cases. So even when the retrieval did its job, the score reads as low because most chunks in the bundle aren't on-topic for the specific question. This isn't a bug. It's a feature of the metric.
 - F — this is the actual problem
Negative mean (0.968) is HIGHER than positive mean (0.941).
That means Faithfulness is scoring shuffled-context cases as more faithful than properly-paired cases.

### Diagnostic

In [ ]:
# Find first positive and first negative
labels_arr = np.array(f_labels_aligned)
scores_arr = np.array(f_scores)

pos_idx = np.where(labels_arr == 1)[0][0]
neg_idx = np.where(labels_arr == 0)[0][0]

print(f"Positive case (idx {pos_idx}, score {scores_arr[pos_idx]:.3f}):")
print(f"  input: {f_cases[pos_idx].input[:120]}")
print(f"  actual_output: {f_cases[pos_idx].actual_output[:200]}")
print(f"  retrieval_context[0]: {f_cases[pos_idx].retrieval_context[0][:200]}")
print()
print(f"Negative case (idx {neg_idx}, score {scores_arr[neg_idx]:.3f}):")
print(f"  input: {f_cases[neg_idx].input[:120]}")
print(f"  actual_output: {f_cases[neg_idx].actual_output[:200]}")
print(f"  retrieval_context[0]: {f_cases[neg_idx].retrieval_context[0][:200]}")

Positive case (idx 0, score 1.000):

input: Where can I find info about support orgs and shelters for UM, incl. local options?

actual_output: You can find information about support organizations and shelters for undocumented migrants (UM), 
including local options, on the Stichting LOS website: https://www.stichtinglos.nl/content/organisatie

retrieval_context[0]: Document: What are the steps I should take to register a child travelling alone from 
Ukraine?  

Unaccompanied minor foreingers are children who are under the age of 18 when they arrive in the Netherl

Negative case (idx 83, score 1.000):

input: Where can I find info about support orgs and shelters for UM, incl. local options?

actual_output: You can find information about support organizations and shelters for undocumented migrants (UM), 
including local options, on the Stichting LOS website: https://www.stichtinglos.nl/content/organisatie

retrieval_context[0]: Document: Is dental care reimbursed?  

This depends on your situation.  

Below is an overview of dental services that are covered by the insurance for Ukrainian displaced persons (RMO):   
Conditions

In both cases, the answer and context are unrelated. That's why F scores them both 1.0 — F asks "does the answer contradict the context?" and an answer about LOS shelters doesn't contradict a child-registration text or a dental-care text. Nothing to flag as unfaithful in either. So labels aren't flipped.

F validation requires positive controls where actual_output is genuinely grounded in retrieval_context. Our dataset lacks such controls because (a) synthesizer contexts are too broad to enable meaningful F testing, and (b) HIA's actual retrievals frequently failed to pull relevant chunks for the synthesizer-generated questions. We document this as a methodological limitation and recommend future work generate matched (input, faithful-answer, narrow-context) triples through targeted prompting or human annotation.

In [59]:
from sklearn.metrics import roc_auc_score

ar_auc = roc_auc_score(ar_labels_aligned, ar_scores)
cr_auc = roc_auc_score(cr_labels_aligned, cr_scores)

print(f"AR AUC: {ar_auc:.3f}  {'PASS' if ar_auc > 0.8 else 'FAIL'} (threshold 0.8)")
print(f"CR AUC: {cr_auc:.3f}  {'PASS' if cr_auc > 0.8 else 'FAIL'} (threshold 0.8)")

AR AUC: 0.690  FAIL (threshold 0.8)

CR AUC: 0.676  FAIL (threshold 0.8)

Pre-committed threshold of AUC > 0.8 was not met. AR and CR achieved AUCs of 0.69 and 0.68 respectively, indicating significant but insufficient discrimination under the construct validation design.

In [58]:
# Find AR negatives that scored 1.0
labels_arr = np.array(ar_labels_aligned)
scores_arr = np.array(ar_scores)

high_neg_idxs = np.where((labels_arr == 0) & (scores_arr >= 0.95))[0]
print(f"AR negatives with score >= 0.95: {len(high_neg_idxs)}")

for idx in high_neg_idxs[:3]:
    print(f"\n--- AR negative idx {idx}, score {scores_arr[idx]:.2f} ---")
    print(f"input: {ar_cases[idx].input[:150]}")
    print(f"actual_output (shuffled from another row): {ar_cases[idx].actual_output[:300]}")

AR negatives with score >= 0.95: 21

--- AR negative idx 39, score 1.00 ---

input: Where can I find info about support orgs and shelters for UM, incl. local options?

actual_output (shuffled from another row): You can find information about local shelter options for UM 
(undocumented migrants) on the following websites:

- Red Cross Helpful Information: https://helpfulinformation.redcross.nl/
- LOS Foundation: https://www.stichtinglos.nl/noodopvang

These sites provide details about available shelters, inc

--- AR negative idx 41, score 1.00 ---

input: How do limited rights, lack of benefits, and housing restrictions increase UM vulnerability to exploitation?

actual_output (shuffled from another row): Limited rights to social benefits and health insurance make undocumented
migrants (UM) more vulnerable to exploitation because they cannot access basic support or protection from the 
government. Without social benefits, they have no financial safety net, making them dependent on informal or unsafe
w

--- AR negative idx 46, score 1.00 ---

input: Where can I find info on NL local shelter options for UM, incl. Red Cross, LOS, and HIA?

actual_output (shuffled from another row): You can find information about shelters for UM (undocumented migrants) 
in the Netherlands, including city-specific options, on the Red Cross Helpful Information website: 
https://helpfulinformation.redcross.nl/ and on the LOS Foundation website: https://www.stichtinglos.nl/noodopvang. 
These sites pro

This means that the AUC isn't measuring "is AR a valid metric?" It's measuring "can AR distinguish between genuinely good answers and also genuinely good answers that happen to come from a different row?".
This is a construct validation design flaw, not a metric flaw. AR is doing its job: when a shuffled answer happens to also answer the question, AR correctly says "yes, this is relevant." The label says "no, this is negative" — but the label is wrong.
The 5–25 percentile band was supposed to capture "non-trivial but still distinguishable" negatives. In practice, on a dataset where many questions are about the same FAQ topics and many answers share the same URLs/organizations, "non-trivial" shaded into "actually valid."

In [62]:
# Find CR negatives that scored 1.0
labels_arr = np.array(cr_labels_aligned)
scores_arr = np.array(cr_scores)

high_neg_idxs = np.where((labels_arr == 0) & (scores_arr >= 0.95))[0]
print(f"CR negatives with score >= 0.95: {len(high_neg_idxs)}")

for idx in high_neg_idxs[:3]:
    print(f"\n--- CR negative idx {idx}, score {scores_arr[idx]:.2f} ---")
    print(f"input: {ar_cases[idx].input[:150]}")
    print(f"actual_output (shuffled from another row): {cr_cases[idx].actual_output[:300]}")

CR negatives with score >= 0.95: 0

For CR, zero negatives scored ≥ 0.95. CR is doing something cleaner. CR's positive mean was 0.262 and negative mean 0.161 — both low. CR scores almost everything in the 0.0–0.3 range because most chunks in any 20-chunk bundle aren't directly relevant to a specific question.

So CR isn't failing because of label noise like AR. CR is failing because the metric compresses everything into a narrow band of low scores, which makes ranking unstable. Positives and negatives both score low; positives just score slightly higher on average.